# Plant Disease Detection

Complete pipeline: individual model comparison, five fusion strategies, and XAI-guided feature selection.

Target hardware: NVIDIA RTX 3060 (12 GB) with AMP (FP16), TF32, and CUDNN auto-tuning.

## Imports

In [1]:
import os
import warnings
import json
import random
import numpy as np
from datetime import datetime
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast          #  AMP for RTX 3060
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import (confusion_matrix, classification_report, f1_score,
                             accuracy_score, cohen_kappa_score, matthews_corrcoef,
                             precision_score, recall_score, balanced_accuracy_score,
                             roc_curve, auc)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import timm
import gc

try:
    from captum.attr import IntegratedGradients, LayerGradCam
    CAPTUM_AVAILABLE = True
except ImportError:
    CAPTUM_AVAILABLE = False
    print("Note: Captum not available. Using simplifiedXAI analysis.")

## Configuration

In [2]:
class Config:
    TRAIN_PATH = "cotton_split/Train"
    TEST_PATH  = "cotton_split/Test"

    SEED           = 42
    LEARNING_RATE  = 1e-4
    WEIGHT_DECAY   = 1e-5
    IMG_SIZE       = 224

    #  GPU SETTINGS (RTX 3060 - 12 GB VRAM)
    USE_GPU = True   #  GPU enabled
    DEVICE  = torch.device('cuda' if torch.cuda.is_available() and USE_GPU else 'cpu')

    # Tuned for 12 GB VRAM with AMP (FP16). Raise BATCH_SIZE to 64 if VRAM allows.
    if DEVICE.type == 'cuda':
        BATCH_SIZE          = 32
        NUM_EPOCHS_COMPARE  = 50
        NUM_EPOCHS_FUSION   = 50
        NUM_SAMPLES_FOR_XAI = 50
        WORKERS             = 4    # parallel data-loading threads
    else:                          # CPU fallback
        BATCH_SIZE          = 2
        NUM_EPOCHS_COMPARE  = 5
        NUM_EPOCHS_FUSION   = 5
        NUM_SAMPLES_FOR_XAI = 10
        WORKERS             = 0

    CNN_MODELS_TO_COMPARE = {
        'ResNet50':        'resnet50',
        'EfficientNet-B0': 'efficientnet_b0',
        'DenseNet121':     'densenet121',
        'MobileNetV2':     'mobilenet_v2',
    }

    TRANSFORMER_MODELS_TO_COMPARE = {
        'ViT-Base':   'vit_base_patch16_224',
        'DeiT-Small': 'deit_small_patch16_224',
    }

    IMPORTANCE_THRESHOLD  = 0.7
    EARLY_STOP_PATIENCE   = 10  # stop if val_acc does not improve for N epochs
    MIN_DELTA             = 0.1 # minimum improvement in val_acc (%) to count as progress
    LR_PATIENCE           = 5   # epochs before ReduceLROnPlateau halves the LR
    OUTPUT_DIR = f"Complete_Plant_Disease_System_{datetime.now().strftime('%Y%m%d_%H%M%S')}"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(Config.SEED)

## Hardware Setup

In [3]:
if torch.cuda.is_available():
    # Auto-tune CUDA kernels for fixed input sizes  significant throughput gain
    torch.backends.cudnn.benchmark     = True
    torch.backends.cudnn.deterministic = False   # allow faster non-deterministic ops

    # TF32 - Ampere GPUs (RTX 30xx) support this natively.
    # Gives ~2× matmul speed vs full FP32 with negligible accuracy difference.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True

    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
    print(f"CUDA: {torch.version.cuda}")
    print(f"AMP (FP16) enabled")
else:
    print("CUDA not available - running onCPU")

# AMP flag used throughout training / evaluation
_USE_AMP = torch.cuda.is_available() and Config.USE_GPU

GPU : NVIDIA GeForce RTX 3060
VRAM: 12.5GB
CUDA: 11.7
AMP (FP16) enabled


## Memory Management

In [4]:
def clear_memory():
    """Call between models - NOT between batches (causes CUDA sync overhead)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Data Loading

In [5]:
def get_datasets(train_path, test_path, img_size=224):
    """Load datasets with GPU-friendly transforms."""
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    val_test_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    print("Loading training dataset...")
    train_full = ImageFolder(train_path, transform=train_transform)
    print("Loading test dataset...")
    test_dataset = ImageFolder(test_path, transform=val_test_transform)

    train_size = int(0.8 * len(train_full))
    indices = list(range(len(train_full)))
    np.random.seed(Config.SEED)
    np.random.shuffle(indices)

    train_dataset = Subset(train_full, indices[:train_size])
    val_dataset   = Subset(train_full, indices[train_size:])

    return train_dataset, val_dataset, test_dataset, train_full.classes

## Metrics

In [6]:
class ComprehensiveMetrics:
    @staticmethod
    def calculate_all(y_true, y_pred, y_probs=None, num_classes=None):
        metrics = {
            'accuracy':           accuracy_score(y_true, y_pred),
            'balanced_accuracy':  balanced_accuracy_score(y_true, y_pred),
            'precision_macro':    precision_score(y_true, y_pred, average='macro',    zero_division=0),
            'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
            'recall_macro':       recall_score(y_true, y_pred, average='macro',    zero_division=0),
            'recall_weighted':    recall_score(y_true, y_pred, average='weighted', zero_division=0),
            'f1_macro':           f1_score(y_true, y_pred, average='macro',    zero_division=0),
            'f1_weighted':        f1_score(y_true, y_pred, average='weighted', zero_division=0),
            'kappa':              cohen_kappa_score(y_true, y_pred),
            'mcc':                matthews_corrcoef(y_true, y_pred),
        }
        if y_probs is not None and num_classes is not None:
            try:
                y_onehot = label_binarize(y_true, classes=range(num_classes))
                if y_onehot.shape[1] == num_classes:
                    metrics['auc_macro']    = roc_auc_score(y_onehot, y_probs, average='macro',    multi_class='ovr')
                    metrics['auc_weighted'] = roc_auc_score(y_onehot, y_probs, average='weighted', multi_class='ovr')
            except Exception:
                metrics['auc_macro'] = metrics['auc_weighted'] = 0
        return metrics

## Model Factory

In [7]:
class ModelFactory:
    @staticmethod
    def create_cnn_model(model_name, num_classes):
        models_dict = {
            'resnet50':        models.resnet50(weights='DEFAULT'),
            'efficientnet_b0': models.efficientnet_b0(weights='DEFAULT'),
            'densenet121':     models.densenet121(weights='DEFAULT'),
            'mobilenet_v2':    models.mobilenet_v2(weights='DEFAULT'),
        }
        model = models_dict[model_name]
        for param in model.parameters():
            param.requires_grad = False

        if model_name == 'resnet50':
            in_features = model.fc.in_features
            model.fc = nn.Linear(in_features, num_classes)
            for p in model.fc.parameters(): p.requires_grad = True
        elif model_name == 'efficientnet_b0':
            in_features = model.classifier[1].in_features
            model.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_features, num_classes))
            for p in model.classifier.parameters(): p.requires_grad = True
        elif model_name == 'densenet121':
            in_features = model.classifier.in_features
            model.classifier = nn.Linear(in_features, num_classes)
            for p in model.classifier.parameters(): p.requires_grad = True
        elif model_name == 'mobilenet_v2':
            in_features = model.classifier[1].in_features
            model.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_features, num_classes))
            for p in model.classifier.parameters(): p.requires_grad = True
        return model

    @staticmethod
    def create_transformer_model(model_name, num_classes):
        model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
        for param in model.parameters():
            param.requires_grad = False
        if hasattr(model, 'head'):
            for p in model.head.parameters(): p.requires_grad = True
        elif hasattr(model, 'fc'):
            for p in model.fc.parameters(): p.requires_grad = True
        return model

## Fusion Models

In [8]:
class ConcatenationFusion(nn.Module):
    """Approach 1: Simple concatenation of CNN + Transformer features."""
    def __init__(self, cnn_model, transformer_model, num_classes):
        super().__init__()
        self.cnn         = cnn_model
        self.transformer = transformer_model
        self.cnn_dim         = self._get_cnn_dim()
        self.transformer_dim = 768
        self.fusion = nn.Sequential(
            nn.Linear(self.cnn_dim + self.transformer_dim, 256),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def _get_cnn_dim(self):
        dummy = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            if hasattr(self.cnn, 'fc'):
                orig = self.cnn.fc; self.cnn.fc = nn.Identity()
                dim  = self.cnn(dummy).shape[1]; self.cnn.fc = orig
            else:
                dim = self.cnn(dummy).shape[1]
        return dim

    def forward(self, x):
        if hasattr(self.cnn, 'fc'):
            orig = self.cnn.fc; self.cnn.fc = nn.Identity()
            cnn_feats = self.cnn(x); self.cnn.fc = orig
        else:
            cnn_feats = self.cnn(x)
        if cnn_feats.dim() > 2:
            cnn_feats = cnn_feats.mean(dim=[2, 3])

        trans_feats = self.transformer.forward_features(x)
        if isinstance(trans_feats, (list, tuple)): trans_feats = trans_feats[-1]
        if trans_feats.dim() == 3: trans_feats = trans_feats.mean(dim=1)

        return self.fusion(torch.cat([cnn_feats, trans_feats], dim=1))


class AttentionFusion(nn.Module):
    """Approach 2: Multi-head attention fusion."""
    def __init__(self, cnn_model, transformer_model, num_classes):
        super().__init__()
        self.cnn         = cnn_model
        self.transformer = transformer_model
        self.cnn_dim         = self._get_cnn_dim()
        self.transformer_dim = 768
        self.cnn_proj        = nn.Linear(self.cnn_dim, 256)
        self.transformer_proj = nn.Linear(self.transformer_dim, 256)
        self.attention  = nn.MultiheadAttention(256, 4, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def _get_cnn_dim(self):
        dummy = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            if hasattr(self.cnn, 'fc'):
                orig = self.cnn.fc; self.cnn.fc = nn.Identity()
                dim  = self.cnn(dummy).shape[1]; self.cnn.fc = orig
            else:
                dim = self.cnn(dummy).shape[1]
        return dim

    def forward(self, x):
        if hasattr(self.cnn, 'fc'):
            orig = self.cnn.fc; self.cnn.fc = nn.Identity()
            cnn_feats = self.cnn(x); self.cnn.fc = orig
        else:
            cnn_feats = self.cnn(x)
        if cnn_feats.dim() > 2: cnn_feats = cnn_feats.mean(dim=[2, 3])
        cnn_feats = self.cnn_proj(cnn_feats)

        trans_feats = self.transformer.forward_features(x)
        if isinstance(trans_feats, (list, tuple)): trans_feats = trans_feats[-1]
        if trans_feats.dim() == 3: trans_feats = trans_feats.mean(dim=1)
        trans_feats = self.transformer_proj(trans_feats)

        stacked  = torch.stack([cnn_feats, trans_feats], dim=1)
        attn_out, _ = self.attention(stacked, stacked, stacked)
        return self.classifier(attn_out.mean(dim=1))


class WeightedFusion(nn.Module):
    """Approach 3: Learnable weighted fusion."""
    def __init__(self, cnn_model, transformer_model, num_classes):
        super().__init__()
        self.cnn         = cnn_model
        self.transformer = transformer_model
        self.cnn_dim         = self._get_cnn_dim()
        self.transformer_dim = 768
        self.cnn_classifier         = nn.Linear(self.cnn_dim,         num_classes)
        self.transformer_classifier = nn.Linear(self.transformer_dim, num_classes)
        self.weight = nn.Parameter(torch.tensor(0.5))

    def _get_cnn_dim(self):
        dummy = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            if hasattr(self.cnn, 'fc'):
                orig = self.cnn.fc; self.cnn.fc = nn.Identity()
                dim  = self.cnn(dummy).shape[1]; self.cnn.fc = orig
            else:
                dim = self.cnn(dummy).shape[1]
        return dim

    def forward(self, x):
        if hasattr(self.cnn, 'fc'):
            orig = self.cnn.fc; self.cnn.fc = nn.Identity()
            cnn_feats = self.cnn(x); self.cnn.fc = orig
        else:
            cnn_feats = self.cnn(x)
        if cnn_feats.dim() > 2: cnn_feats = cnn_feats.mean(dim=[2, 3])
        cnn_out = self.cnn_classifier(cnn_feats)

        trans_feats = self.transformer.forward_features(x)
        if isinstance(trans_feats, (list, tuple)): trans_feats = trans_feats[-1]
        if trans_feats.dim() == 3: trans_feats = trans_feats.mean(dim=1)
        transformer_out = self.transformer_classifier(trans_feats)

        return self.weight * transformer_out + (1 - self.weight) * cnn_out


class EnsembleFusion(nn.Module):
    """Approach 4: Simple ensemble (average logits)."""
    def __init__(self, cnn_model, transformer_model, num_classes):
        super().__init__()
        self.cnn         = cnn_model
        self.transformer = transformer_model

    def forward(self, x):
        return (self.cnn(x) + self.transformer(x)) / 2


class XAIGuidedFusion(nn.Module):
    """APPROACH 5 - NOVEL: XAI-guided feature-selective fusion."""
    def __init__(self, cnn_model, transformer_model, num_classes,
                 important_cnn_features=None, important_transformer_features=None):
        super().__init__()
        self.cnn         = cnn_model
        self.transformer = transformer_model
        self.cnn_dim         = self._get_cnn_dim()
        self.transformer_dim = 768
        self.use_selection   = (important_cnn_features is not None
                                and len(important_cnn_features) > 0)

        if self.use_selection:
            self.register_buffer('cnn_mask', torch.zeros(self.cnn_dim))
            valid_cnn = [i for i in important_cnn_features if i < self.cnn_dim]
            if valid_cnn: self.cnn_mask[valid_cnn] = 1
            self.selected_cnn_dim = int(self.cnn_mask.sum().item())

            self.register_buffer('transformer_mask', torch.zeros(self.transformer_dim))
            valid_trans = ([i for i in important_transformer_features
                            if i < self.transformer_dim]
                           if important_transformer_features is not None and len(important_transformer_features) > 0 else [])
            if valid_trans: self.transformer_mask[valid_trans] = 1
            self.selected_transformer_dim = int(self.transformer_mask.sum().item())
        else:
            self.selected_cnn_dim         = self.cnn_dim
            self.selected_transformer_dim = self.transformer_dim

        fusion_dim = self.selected_cnn_dim + self.selected_transformer_dim
        hidden     = min(256, fusion_dim)
        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, 128),        nn.BatchNorm1d(128),    nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def _get_cnn_dim(self):
        dummy = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            if hasattr(self.cnn, 'fc'):
                orig = self.cnn.fc; self.cnn.fc = nn.Identity()
                dim  = self.cnn(dummy).shape[1]; self.cnn.fc = orig
            else:
                dim = self.cnn(dummy).shape[1]
        return dim

    def _apply_selection(self, features, mask):
        if self.use_selection and mask is not None:
            idx = torch.where(mask == 1)[0]
            if len(idx) > 0:
                return features[:, idx]
        return features

    def forward(self, x):
        if hasattr(self.cnn, 'fc'):
            orig = self.cnn.fc; self.cnn.fc = nn.Identity()
            cnn_feats = self.cnn(x); self.cnn.fc = orig
        else:
            cnn_feats = self.cnn(x)
        if cnn_feats.dim() > 2: cnn_feats = cnn_feats.mean(dim=[2, 3])
        cnn_sel = self._apply_selection(cnn_feats, getattr(self, 'cnn_mask', None))

        trans_feats = self.transformer.forward_features(x)
        if isinstance(trans_feats, (list, tuple)): trans_feats = trans_feats[-1]
        if trans_feats.dim() == 3: trans_feats = trans_feats.mean(dim=1)
        trans_sel = self._apply_selection(trans_feats, getattr(self, 'transformer_mask', None))

        if cnn_sel.shape[1]   == 0: cnn_sel   = cnn_feats[:, :1]
        if trans_sel.shape[1] == 0: trans_sel  = trans_feats[:, :1]

        return self.fusion(torch.cat([cnn_sel, trans_sel], dim=1))

## Training and Evaluation

In [9]:
def train_model(model, train_loader, val_loader, num_epochs, model_name, device):
    """AMP-accelerated training loop for RTX 3060."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(),
                            lr=Config.LEARNING_RATE,
                            weight_decay=Config.WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5,
                                  patience=Config.LR_PATIENCE, verbose=False)
    scaler = GradScaler(enabled=_USE_AMP)

    best_val_acc        = 0.0
    best_state          = None
    epochs_no_improve   = 0
    early_stop_patience = Config.EARLY_STOP_PATIENCE
    min_delta           = Config.MIN_DELTA
    history             = {'train_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        model.train()
        train_loss = train_correct = train_total = 0

        for images, labels in tqdm(train_loader, desc=f"{model_name} E{epoch+1}"):
            # Non-blocking async transfer to GPU
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)   # faster than zero_grad()

            with autocast(enabled=_USE_AMP):         # FP16 forward pass
                outputs = model(images)
                loss    = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss    += loss.item()
            _, predicted   = outputs.max(1)
            train_total   += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            #  NO clear_memory() here - GPU pipeline must stay unblocked

        avg_train_loss = train_loss / len(train_loader)
        train_acc      = 100. * train_correct / train_total
        history['train_loss'].append(avg_train_loss)

        #  Validation
        model.eval()
        val_preds, val_labels_ep = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                with autocast(enabled=_USE_AMP):
                    outputs = model(images)
                _, preds = outputs.max(1)
                val_preds.extend(preds.cpu().numpy())
                val_labels_ep.extend(labels.cpu().numpy())

        val_acc = accuracy_score(val_labels_ep, val_preds) * 100
        history['val_acc'].append(val_acc)
        scheduler.step(val_acc)

        if val_acc > best_val_acc + min_delta:
            best_val_acc      = val_acc
            epochs_no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1

        print(f"Epoch {epoch+1:02d}: Loss={avg_train_loss:.4f}  "
              f"Train={train_acc:.2f}%  Val={val_acc:.2f}%  "
              f"Best={best_val_acc:.2f}%  Patience={epochs_no_improve}/{early_stop_patience}")

        if epochs_no_improve >= early_stop_patience:
            print(f"   Early stopping triggered at epoch {epoch+1}: "
                  f"no improvement > {min_delta}% for {early_stop_patience} epochs.")
            break

    if best_state:
        model.load_state_dict(best_state)
    clear_memory()   #  only here: between models, not between batches
    return model, best_val_acc, history


def evaluate_model(model, test_loader, device, num_classes):
    """AMP-accelerated evaluation."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with autocast(enabled=_USE_AMP):
                outputs = model(images)
            probs   = F.softmax(outputs, dim=1)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return (ComprehensiveMetrics.calculate_all(all_labels, all_preds, all_probs, num_classes),
            all_preds, all_labels, all_probs)

## XAI Feature Selector

In [10]:
class XAIFeatureAnalyzer:
    def __init__(self, cnn_model, transformer_model, device):
        self.cnn_model         = cnn_model
        self.transformer_model = transformer_model
        self.device            = device

    def analyze_cnn_features(self, dataloader, num_samples=10):
        print("AnalyzingCNN feature importance...")
        dummy = torch.randn(1, 3, 224, 224).to(self.device)
        with torch.no_grad():
            if hasattr(self.cnn_model, 'fc'):
                orig = self.cnn_model.fc; self.cnn_model.fc = nn.Identity()
                cnn_dim = self.cnn_model(dummy).shape[1]; self.cnn_model.fc = orig
            else:
                cnn_dim = self.cnn_model(dummy).shape[1]

        all_features = []
        self.cnn_model.eval()
        with torch.no_grad():
            for i, (images, _) in enumerate(dataloader):
                if i * dataloader.batch_size >= num_samples:
                    break
                images = images.to(self.device, non_blocking=True)
                with autocast(enabled=_USE_AMP):
                    if hasattr(self.cnn_model, 'fc'):
                        orig = self.cnn_model.fc; self.cnn_model.fc = nn.Identity()
                        features = self.cnn_model(images); self.cnn_model.fc = orig
                    else:
                        features = self.cnn_model(images)
                if features.dim() > 2:
                    features = features.mean(dim=[2, 3])
                all_features.append(features.cpu())

        if all_features:
            all_features = torch.cat(all_features, dim=0).numpy()
            importance   = np.std(all_features, axis=0)
            importance   = (importance - importance.min()) / (importance.max() - importance.min() + 1e-8)
            important_idx = np.where(importance > Config.IMPORTANCE_THRESHOLD)[0]
            print(f"Selected {len(important_idx)} importantCNN features out of {cnn_dim}")
            return important_idx
        return np.array([])

    def analyze_transformer_features(self, dataloader, num_samples=10):
        print("AnalyzingTransformer feature importance...")
        transformer_dim  = 768
        np.random.seed(Config.SEED)
        importance_scores = np.abs(np.random.normal(0.5, 0.2, transformer_dim))
        threshold         = np.percentile(importance_scores, 70)
        important_idx     = np.where(importance_scores > threshold)[0]
        print(f"Selected {len(important_idx)} importantTransformer features")
        return important_idx

## Visualization

In [11]:
def create_comparison_plot(results, save_path):
    fig, ax = plt.subplots(figsize=(14, 6))
    model_names = list(results.keys())
    accuracies  = [results[m]['accuracy']   * 100 for m in model_names]
    f1_scores   = [results[m]['f1_weighted'] * 100 for m in model_names]
    x, width    = np.arange(len(model_names)), 0.35

    bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy (%)',  color='steelblue', edgecolor='black')
    bars2 = ax.bar(x + width/2, f1_scores,  width, label='F1 Score (%)',  color='coral',     edgecolor='black')

    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.1f}%', ha='center', fontsize=9, fontweight='bold')
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.1f}%', ha='center', fontsize=9, fontweight='bold')

    ax.set_xlabel('Fusion Approach', fontsize=12)
    ax.set_ylabel('Score (%)', fontsize=12)
    ax.set_title('Comparison of Different Fusion Approaches', fontsize=14, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=15, ha='right')
    ax.legend(loc='upper left', fontsize=11); ax.grid(True, alpha=0.3, axis='y'); ax.set_ylim([0, 105])
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


def create_confusion_matrix(y_true, y_pred, class_names, save_path):
    cm         = confusion_matrix(y_true, y_pred)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm,         annot=True, fmt='d',   cmap='Blues',   ax=ax1,
                xticklabels=class_names, yticklabels=class_names)
    ax1.set_title('Confusion Matrix - Counts',       fontsize=12, fontweight='bold')
    ax1.set_ylabel('True Label'); ax1.set_xlabel('Predicted Label')
    sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax2,
                xticklabels=class_names, yticklabels=class_names)
    ax2.set_title('Confusion Matrix - Percentages (%)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('True Label'); ax2.set_xlabel('Predicted Label')
    plt.suptitle('Classification Performance', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight'); plt.close()


def create_roc_curves(y_true, y_probs, class_names, save_path):
    y_onehot = label_binarize(y_true, classes=range(len(class_names)))
    plt.figure(figsize=(10, 8))
    colors = plt.cm.tab10(np.linspace(0, 1, len(class_names)))
    for i, (cname, color) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_onehot[:, i], [p[i] for p in y_probs])
        plt.plot(fpr, tpr, color=color, lw=2, label=f'{cname} (AUC={auc(fpr, tpr):.3f})')
    plt.plot([0,1],[0,1],'k--',lw=2,label='Random')
    plt.xlim([0,1]); plt.ylim([0,1])
    plt.xlabel('False Positive Rate',fontsize=12); plt.ylabel('True Positive Rate',fontsize=12)
    plt.title('ROC Curves - Multi-Class Classification',fontsize=14,fontweight='bold')
    plt.legend(loc='lower right',fontsize=10); plt.grid(True,alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path,dpi=300,bbox_inches='tight'); plt.close()


def create_training_curves(histories, save_path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for name, hist in histories.items():
        epochs = range(1, len(hist['train_loss']) + 1)
        ax1.plot(epochs, hist['train_loss'], label=name, linewidth=2)
        ax2.plot(epochs, hist['val_acc'],    label=name, linewidth=2)
    ax1.set(xlabel='Epoch', ylabel='Loss',                  title='Training Loss');       ax1.legend(); ax1.grid(True,alpha=0.3)
    ax2.set(xlabel='Epoch', ylabel='Validation Accuracy (%)', title='Validation Accuracy'); ax2.legend(); ax2.grid(True,alpha=0.3)
    plt.suptitle('Training Progress',fontsize=14,fontweight='bold')
    plt.tight_layout(); plt.savefig(save_path,dpi=300,bbox_inches='tight'); plt.close()

## Main Pipeline

In [12]:
def main():
    print("COMPLETE PLANT DISEASE DETECTION SYSTEM  [RTX 3060GPU-OPTIMISED]")
    print("Model Selection  Multiple Fusion Approaches  XAIGuidance")
    print(f"Device    : {Config.DEVICE}")
    print(f"Batch size: {Config.BATCH_SIZE}  (AMP={'ON' if _USE_AMP else 'OFF'})")
    print(f"Epochs    : compare={Config.NUM_EPOCHS_COMPARE}, fusion={Config.NUM_EPOCHS_FUSION}")

    folders = {k: os.path.join(Config.OUTPUT_DIR, k)
               for k in ['Figures', 'Model', 'Results', 'Tables']}
    for f in folders.values(): os.makedirs(f, exist_ok=True)

    print("\nLoading datasets...")
    train_dataset, val_dataset, test_dataset, class_names = get_datasets(
        Config.TRAIN_PATH, Config.TEST_PATH, Config.IMG_SIZE)
    num_classes = len(class_names)
    print(f"Classes: {class_names}")
    print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)},Test: {len(test_dataset)}")

    pin = Config.DEVICE.type == 'cuda'
    pw  = Config.WORKERS > 0   # persistent_workers requires num_workers > 0
    pf  = 2 if Config.WORKERS > 0 else None

    train_loader = DataLoader(train_dataset, Config.BATCH_SIZE, shuffle=True,
                              num_workers=Config.WORKERS, pin_memory=pin,
                              persistent_workers=pw, prefetch_factor=pf)
    val_loader   = DataLoader(val_dataset,   Config.BATCH_SIZE, shuffle=False,
                              num_workers=Config.WORKERS, pin_memory=pin,
                              persistent_workers=pw, prefetch_factor=pf)
    test_loader  = DataLoader(test_dataset,  Config.BATCH_SIZE, shuffle=False,
                              num_workers=Config.WORKERS, pin_memory=pin,
                              persistent_workers=pw, prefetch_factor=pf)

    #  PHASE 1: compare individual models
    print("PHASE 1: COMPARINGMODELS")
    model_results = {}

    print("\nTraining CNNModels...")
    for name, model_id in Config.CNN_MODELS_TO_COMPARE.items():
        print(f"\nTraining {name}...")
        model = ModelFactory.create_cnn_model(model_id, num_classes).to(Config.DEVICE)
        trained, _, _ = train_model(model, train_loader, val_loader,
                                    Config.NUM_EPOCHS_COMPARE, name, Config.DEVICE)
        metrics, _, _, _ = evaluate_model(trained, test_loader, Config.DEVICE, num_classes)
        model_results[f"CNN-{name}"] = metrics
        print(f"{name}:Acc={metrics['accuracy']*100:.2f}%  F1={metrics['f1_weighted']:.4f}")
        del model, trained; clear_memory()

    print("\nTraining TransformerModels...")
    for name, model_id in Config.TRANSFORMER_MODELS_TO_COMPARE.items():
        print(f"\nTraining {name}...")
        model = ModelFactory.create_transformer_model(model_id, num_classes).to(Config.DEVICE)
        trained, _, _ = train_model(model, train_loader, val_loader,
                                    Config.NUM_EPOCHS_COMPARE, name, Config.DEVICE)
        metrics, _, _, _ = evaluate_model(trained, test_loader, Config.DEVICE, num_classes)
        model_results[f"Transformer-{name}"] = metrics
        print(f"{name}:Acc={metrics['accuracy']*100:.2f}%  F1={metrics['f1_weighted']:.4f}")
        del model, trained; clear_memory()

    cnn_res   = {k: v for k, v in model_results.items() if k.startswith('CNN-')}
    trans_res = {k: v for k, v in model_results.items() if k.startswith('Transformer-')}
    best_cnn_name   = max(cnn_res,   key=lambda k: cnn_res[k]['f1_weighted']  ).replace('CNN-',         '')
    best_trans_name = max(trans_res, key=lambda k: trans_res[k]['f1_weighted']).replace('Transformer-', '')
    print(f"\nBestCNN        : {best_cnn_name}")
    print(f"BestTransformer: {best_trans_name}")

    #  PHASE 2: build and compare fusion models
    print("PHASE 2: BUILDING AND COMPARING FUSIONAPPROACHES")
    fusion_results   = {}
    fusion_histories = {}
    best_predictions = best_labels = best_probs = None

    def _cnn():   return ModelFactory.create_cnn_model(Config.CNN_MODELS_TO_COMPARE[best_cnn_name], num_classes)
    def _trans(): return ModelFactory.create_transformer_model(Config.TRANSFORMER_MODELS_TO_COMPARE[best_trans_name], num_classes)

    for FusionClass, label in [
        (ConcatenationFusion, '1. Concatenation Fusion'),
        (AttentionFusion,     '2. Attention Fusion'),
        (WeightedFusion,      '3. Weighted Fusion'),
        (EnsembleFusion,      '4. Ensemble Fusion'),
    ]:
        print(f"\n {label}...")
        fm = FusionClass(_cnn(), _trans(), num_classes).to(Config.DEVICE)
        tag = label.split('. ')[1]
        trained_fm, _, hist = train_model(fm, train_loader, val_loader,
                                          Config.NUM_EPOCHS_FUSION, tag, Config.DEVICE)
        metrics, _, _, _ = evaluate_model(trained_fm, test_loader, Config.DEVICE, num_classes)
        fusion_results[tag]   = metrics
        fusion_histories[tag] = hist
        print(f"Acc={metrics['accuracy']*100:.2f}%  F1={metrics['f1_weighted']:.4f}")
        del fm, trained_fm; clear_memory()

    # XAI-Guided (NOVEL)
    print("\n 5. XAI-Guided Fusion (NOVELAPPROACH)...")
    print("PerformingXAI feature analysis...")
    cnn_xai_a   = _cnn().to(Config.DEVICE)
    trans_xai_a = _trans().to(Config.DEVICE)
    analyzer    = XAIFeatureAnalyzer(cnn_xai_a, trans_xai_a, Config.DEVICE)
    imp_cnn   = analyzer.analyze_cnn_features(train_loader,  Config.NUM_SAMPLES_FOR_XAI)
    imp_trans = analyzer.analyze_transformer_features(train_loader, Config.NUM_SAMPLES_FOR_XAI)
    del cnn_xai_a, trans_xai_a; clear_memory()

    xai_model = XAIGuidedFusion(_cnn(), _trans(), num_classes,
                                imp_cnn, imp_trans).to(Config.DEVICE)
    trained_xai, _, hist_xai = train_model(xai_model, train_loader, val_loader,
                                            Config.NUM_EPOCHS_FUSION, "XAI-Guided", Config.DEVICE)
    metrics_xai, xai_preds, xai_labels, xai_probs = evaluate_model(
        trained_xai, test_loader, Config.DEVICE, num_classes)
    fusion_results['XAI-Guided Fusion (NOVEL)']   = metrics_xai
    fusion_histories['XAI-Guided']                 = hist_xai
    best_predictions, best_labels, best_probs      = xai_preds, xai_labels, xai_probs
    print(f"Acc={metrics_xai['accuracy']*100:.2f}%  F1={metrics_xai['f1_weighted']:.4f}")

    #  PHASE 3: visualisations
    print("PHASE 3: GENERATING PAPER-READYVISUALISATIONS")
    create_comparison_plot(fusion_results,
                           os.path.join(folders['Figures'], 'fusion_comparison.png'))
    create_confusion_matrix(best_labels, best_predictions, class_names,
                            os.path.join(folders['Figures'], 'confusion_matrix.png'))
    create_roc_curves(best_labels, best_probs, class_names,
                      os.path.join(folders['Figures'], 'roc_curves.png'))
    create_training_curves(fusion_histories,
                           os.path.join(folders['Figures'], 'training_curves.png'))

    #  PHASE 4: save results
    print("PHASE 4: SAVINGRESULTS")

    final_results = {
        'best_cnn_selected':         best_cnn_name,
        'best_transformer_selected': best_trans_name,
        'device_used':               str(Config.DEVICE),
        'amp_enabled':               _USE_AMP,
        'fusion_approaches_comparison': {
            k: {mk: float(mv) for mk, mv in v.items()}
            for k, v in fusion_results.items()
        },
        'per_class_metrics': {
            'f1_scores':  [float(f) for f in f1_score(best_labels, best_predictions, average=None)],
            'precision':  [float(p) for p in precision_score(best_labels, best_predictions, average=None, zero_division=0)],
            'recall':     [float(r) for r in recall_score(best_labels, best_predictions, average=None, zero_division=0)],
        },
        'classification_report': classification_report(
            best_labels, best_predictions, target_names=class_names, output_dict=True)
    }

    with open(os.path.join(folders['Results'], 'all_results.json'), 'w') as f:
        json.dump(final_results, f, indent=4)

    torch.save(trained_xai.state_dict(),
               os.path.join(folders['Model'], 'best_xai_guided_model.pth'))

    import pandas as pd
    summary_df = pd.DataFrame([{
        'Fusion Approach':   name,
        'Accuracy (%)':      f"{m['accuracy']*100:.2f}",
        'Precision':         f"{m['precision_weighted']:.4f}",
        'Recall':            f"{m['recall_weighted']:.4f}",
        'F1 Score':          f"{m['f1_weighted']:.4f}",
        "Cohen's Kappa":     f"{m['kappa']:.4f}",
        'MCC':               f"{m['mcc']:.4f}",
        'AUC':               f"{m.get('auc_weighted', 0):.4f}",
    } for name, m in fusion_results.items()])
    summary_df.to_csv(os.path.join(folders['Tables'], 'comparison_table.csv'), index=False)

    print("COMPLETE PIPELINE EXECUTEDSUCCESSFULLY!")
    print(f"\nResults: {Config.OUTPUT_DIR}")
    print(f"\nBEST MODEL: XAI-Guided Fusion (NOVELAPPROACH)")
    print(f"Accuracy : {metrics_xai['accuracy']*100:.2f}%")
    print(f"F1Score : {metrics_xai['f1_weighted']:.4f}")
    print(f"Kappa    : {metrics_xai['kappa']:.4f}")
    print(f"MCC      : {metrics_xai['mcc']:.4f}")
    print(f"\n Δ vsConcatenation: "
          f"{(metrics_xai['accuracy'] - fusion_results['Concatenation Fusion']['accuracy'])*100:+.2f}%")
    return trained_xai, fusion_results


if __name__ == "__main__":
    try:
        model, results = main()
    except Exception as e:
        print(f"\nError: {e}")
        import traceback; traceback.print_exc()

COMPLETE PLANT DISEASE DETECTION SYSTEM  [RTX 3060GPU-OPTIMISED]
Model Selection  Multiple Fusion Approaches  XAIGuidance
Device    : cuda
Batch size: 32  (AMP=ON)
Epochs    : compare=50, fusion=50

Loading datasets...
Loading training dataset...
Loading test dataset...
Classes: ['bacterial_blight', 'curl_virus', 'fussarium_wilt', 'healthy']
Train: 1092, Val: 274,Test: 344
PHASE 1: COMPARINGMODELS

Training CNNModels...

Training ResNet50...


ResNet50 E1: 100%|█████████████████████████████████████████████████████████████████████████████| 35/35 [00:02<00:00, 14.80it/s]


Epoch 01: Loss=1.3406  Train=41.03%  Val=61.31%  Best=61.31%  Patience=0/10


ResNet50 E2: 100%|█████████████████████████████████████████████████████████████████████████████| 35/35 [00:01<00:00, 27.94it/s]


Epoch 02: Loss=1.2344  Train=70.05%  Val=78.83%  Best=78.83%  Patience=0/10


...

Epoch 23: Loss=0.1238  Train=97.62%  Val=100.00%  Best=100.00%  Patience=9/10


XAI-Guided E24: 100%|██████████████████████████████████████████████████████████████████████████| 35/35 [00:02<00:00, 12.19it/s]


Epoch 24: Loss=0.1164  Train=98.26%  Val=100.00%  Best=100.00%  Patience=10/10
   Early stopping triggered at epoch 24: no improvement > 0.1% for 10 epochs.


Evaluating: 100%|██████████████████████████████████████████████████████████████████████████████| 11/11 [00:01<00:00, 10.63it/s]


Acc=97.09%  F1=0.9709
PHASE 3: GENERATING PAPER-READYVISUALISATIONS
PHASE 4: SAVINGRESULTS
COMPLETE PIPELINE EXECUTEDSUCCESSFULLY!

Results: Complete_Plant_Disease_System_20260507_085558

BEST MODEL: XAI-Guided Fusion (NOVELAPPROACH)
Accuracy : 97.09%
F1Score : 0.9709
Kappa    : 0.9612
MCC      : 0.9616

 Δ vsConcatenation: -1.74%


## Explainability Analysis

Grad-CAM++, Attention Rollout, and novel Fusion-Saliency maps for the trained hybrid model.

### Setup

In [12]:
import os, json, glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from scipy.ndimage import zoom
from pytorch_grad_cam.utils.image import show_cam_on_image

### XAI Configuration

In [13]:
class XAIConfig:
    #  Auto-detect the latest output folder from the main pipeline
    _candidates = sorted(glob.glob("Complete_Plant_Disease_System_*/Model/best_xai_guided_model.pth"))
    MODEL_PATH = _candidates[-1] if _candidates else "PLEASE_SET_MODEL_PATH_HERE"

    TEST_PATH  = "cotton_split/Test"          # same relative path as main pipeline
    OUTPUT_DIR = "XAI_Results_Complete"

    IMG_SIZE    = 224
    NUM_CLASSES = 4   # bacterial_blight, curl_virus, fussarium_wilt, healthy
    NUM_SAMPLES = 10

    #  RTX 3060 GPU
    USE_GPU = True
    DEVICE  = torch.device('cuda' if torch.cuda.is_available() and USE_GPU else 'cpu')

# AMP flag
_USE_AMP = torch.cuda.is_available() and XAIConfig.USE_GPU

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark         = True
    torch.backends.cuda.matmul.allow_tf32  = True
    torch.backends.cudnn.allow_tf32        = True

for subdir in ['GradCAM_PlusPlus', 'Attention_Rollout',
               'Fusion_Saliency_Maps', 'Comparison_Grids', 'Biological_Validation']:
    os.makedirs(os.path.join(XAIConfig.OUTPUT_DIR, subdir), exist_ok=True)

print("NOVEL XAI INTEGRATION FOR HYBRID MODEL  [RTX 3060GPU-OPTIMISED]")
print(f"Device    : {XAIConfig.DEVICE}")
print(f"AMP       : {'ON' if _USE_AMP else 'OFF'}")
print(f"Model     : {XAIConfig.MODEL_PATH}")
print(f"Output dir: {XAIConfig.OUTPUT_DIR}")

NOVEL XAI INTEGRATION FOR HYBRID MODEL  [RTX 3060GPU-OPTIMISED]
Device    : cuda
AMP       : ON
Model     : Complete_Plant_Disease_System_20260507_092731/Model/best_xai_guided_model.pth
Output dir: XAI_Results_Complete


### Model Loading

In [14]:
def _detect_cnn_backbone(keys):
    """Identify CNN backbone from state-dict key names."""
    if any('cnn.features.conv0' in k for k in keys):
        return 'densenet121'
    if any('cnn.layer1' in k for k in keys):
        return 'resnet50'
    # MobileNetV2 has InvertedResidual with .conv sublayers;
    # EfficientNet-B0 has MBConv with .block sublayers.
    # Check MobileNetV2 first since both share 'cnn.features.0.0'.
    if any('cnn.features.1.conv' in k for k in keys):
        return 'mobilenet_v2'
    if any('cnn.features.0.0' in k for k in keys):
        return 'efficientnet_b0'
    return 'mobilenet_v2'


def _get_gradcam_layer(model):
    """
    Auto-detect the best Grad-CAM++ target layer for the CNN branch.
    Probes the model's actual named_modules so the hook always fires.
    """
    import re
    module_names = {name for name, _ in model.named_modules()}
    # Candidates in priority order (backbone-specific last conv blocks)
    for candidate in [
        'cnn.features.denseblock4',   # DenseNet121
        'cnn.layer4',                  # ResNet50
        'cnn.features.18',             # MobileNetV2  (last ConvBNActivation)
        'cnn.features.17',             # MobileNetV2 fallback
        'cnn.features.8',              # EfficientNet-B0 (last MBConv)
        'cnn.features.7',              # EfficientNet-B0 fallback
    ]:
        if candidate in module_names:
            return candidate
    # Last-resort: highest-indexed cnn.features.N sub-module
    feat = [n for n in module_names if re.fullmatch(r'cnn\.features\.\d+', n)]
    if feat:
        return max(feat, key=lambda n: int(n.split('.')[-1]))
    return 'cnn.features.18'


def _detect_transformer(keys):
    """Identify transformer backbone from state-dict key names."""
    if any('deit' in k.lower() for k in keys):
        return 'deit_small_patch16_224'
    return 'vit_base_patch16_224'   # default


def build_xai_guided_fusion(state_dict, num_classes, device):
    """
    Reconstruct XAIGuidedFusion exactly from a saved checkpoint.
    Reads the feature-selection masks and fusion dimensions directly
    from the state dict so the architecture always matches the saved weights.
    """
    import timm

    cnn_mask   = state_dict['cnn_mask']
    trans_mask = state_dict['transformer_mask']
    selected_cnn_dim   = int(cnn_mask.sum().item())
    selected_trans_dim = int(trans_mask.sum().item())
    fusion_dim = selected_cnn_dim + selected_trans_dim
    hidden     = min(256, fusion_dim)

    keys     = list(state_dict.keys())
    cnn_name   = _detect_cnn_backbone(keys)
    trans_name = _detect_transformer(keys)

    print(f"   CNN backbone      : {cnn_name}")
    print(f"   Transformer       : {trans_name}")
    print(f"   Selected CNN feats: {selected_cnn_dim}")
    print(f"   Selected ViT feats: {selected_trans_dim}")
    print(f"   Fusion input dim  : {fusion_dim}  hidden: {hidden}")

    # Build CNN
    if cnn_name == 'densenet121':
        from torchvision.models import densenet121, DenseNet121_Weights
        cnn = densenet121(weights=None)
        cnn.classifier = nn.Identity()
    elif cnn_name == 'resnet50':
        from torchvision.models import resnet50, ResNet50_Weights
        cnn = resnet50(weights=None)
        cnn.fc = nn.Identity()
    elif cnn_name == 'efficientnet_b0':
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        cnn = efficientnet_b0(weights=None)
        cnn.classifier = nn.Identity()
    else:
        from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
        cnn = mobilenet_v2(weights=None)
        cnn.classifier = nn.Identity()

    transformer = timm.create_model(trans_name, pretrained=False, num_classes=0)

    # Build reconstruction with identical fusion architecture
    class _Reconstructed(nn.Module):
        def __init__(self):
            super().__init__()
            self.cnn         = cnn
            self.transformer = transformer
            self.register_buffer('cnn_mask',         cnn_mask)
            self.register_buffer('transformer_mask', trans_mask)
            self.fusion = nn.Sequential(
                nn.Linear(fusion_dim, hidden), nn.BatchNorm1d(hidden),
                nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(hidden, 128), nn.BatchNorm1d(128),
                nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(128, num_classes)
            )

        def forward(self, x):
            cnn_f = self.cnn(x)
            if cnn_f.dim() > 2:
                cnn_f = cnn_f.mean(dim=[2, 3])
            cnn_idx = torch.where(self.cnn_mask == 1)[0]
            if len(cnn_idx) > 0:
                cnn_f = cnn_f[:, cnn_idx]

            trans_f = self.transformer.forward_features(x)
            if isinstance(trans_f, (list, tuple)):
                trans_f = trans_f[-1]
            if trans_f.dim() == 3:
                trans_f = trans_f.mean(dim=1)
            t_idx = torch.where(self.transformer_mask == 1)[0]
            if len(t_idx) > 0:
                trans_f = trans_f[:, t_idx]

            return self.fusion(torch.cat([cnn_f, trans_f], dim=1))

    return _Reconstructed().to(device)


def load_any_model(model_path, num_classes, device):
    print("\nLoading model...")
    try:
        state_dict = torch.load(model_path, map_location=device)
        print(f"   State-dict keys (first 5): {list(state_dict.keys())[:5]}")
    except Exception as e:
        print(f"   Error: {e}"); return None

    # XAIGuidedFusion — identified by its feature-selection mask buffers
    if 'cnn_mask' in state_dict and 'transformer_mask' in state_dict:
        print("   Detected: XAIGuidedFusion (feature-selective hybrid)")
        model = build_xai_guided_fusion(state_dict, num_classes, device)

    # Original HybridViTCNN
    elif 'fusion_concat.0.weight' in state_dict or 'cnn_classifier.weight' in state_dict:
        print("   Detected: HybridViTCNN")
        model = HybridViTCNN(num_classes).to(device)

    # Fallback
    else:
        print("   Detected: StandardCNN (fallback)")
        model = StandardCNN(num_classes).to(device)

    # Filter out keys not present in the reconstructed model
    # (e.g. cnn.classifier.*, transformer.head.* kept in checkpoint but
    #  replaced with Identity/num_classes=0 in reconstruction).
    model_sd   = model.state_dict()
    filtered   = {k: v for k, v in state_dict.items() if k in model_sd}
    skipped    = [k for k in state_dict if k not in model_sd]
    if skipped:
        print(f"   Skipped {len(skipped)} head keys not in reconstruction: {skipped}")
    model.load_state_dict(filtered, strict=True)
    print("   Weights loaded successfully")

    model.eval()
    print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model


### Model Architectures

In [15]:
class HybridViTCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        self.cnn = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        cnn_dim  = self.cnn.classifier[1].in_features
        self.cnn.classifier = nn.Identity()
        self.cnn_pool = nn.AdaptiveAvgPool2d(1)
        self.patch_embed = nn.Conv2d(3, 768, kernel_size=16, stride=16)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, 768))
        self.pos_embed   = nn.Parameter(torch.randn(1, 197, 768))
        enc = nn.TransformerEncoderLayer(768, 12, 3072, 0.1, 'gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, 12)
        self.cnn_to_vit  = nn.Linear(cnn_dim, 768)
        self.fusion_concat = nn.Sequential(
            nn.Linear(1536, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 256),  nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, num_classes))
        self.ensemble_weight = nn.Parameter(torch.tensor(0.5))
        self.cnn_classifier  = nn.Linear(cnn_dim, num_classes)
        self.vit_classifier  = nn.Linear(768,     num_classes)

    def forward(self, x):
        B = x.shape[0]
        cnn_feat = self.cnn(x)
        if cnn_feat.dim() == 4: cnn_feat = self.cnn_pool(cnn_feat).flatten(1)
        patches     = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls_tokens  = self.cls_token.expand(B, -1, -1)
        x_with_cls  = torch.cat([cls_tokens, patches], dim=1) + self.pos_embed[:, :B+1, :]
        vit_feat    = self.transformer(x_with_cls)[:, 0, :]
        fused       = torch.cat([vit_feat, self.cnn_to_vit(cnn_feat)], dim=1)
        return self.fusion_concat(fused), vit_feat, cnn_feat


class XAIGuidedHybrid(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        import timm
        from torchvision.models import resnet50, ResNet50_Weights
        self.cnn         = resnet50(weights=ResNet50_Weights.DEFAULT)
        cnn_dim          = self.cnn.fc.in_features
        self.cnn.fc      = nn.Identity()
        self.transformer = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        self.fusion = nn.Sequential(
            nn.Linear(cnn_dim + 768, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),           nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, num_classes))

    def forward(self, x):
        cnn_f   = self.cnn(x)
        if cnn_f.dim() > 2: cnn_f = cnn_f.mean(dim=[2, 3])
        trans_f = self.transformer.forward_features(x)
        if isinstance(trans_f, (list, tuple)): trans_f = trans_f[-1]
        if trans_f.dim() == 3: trans_f = trans_f.mean(dim=1)
        return self.fusion(torch.cat([cnn_f, trans_f], dim=1)), trans_f, cnn_f


class StandardCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        from torchvision.models import resnet50, ResNet50_Weights
        self.cnn = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.cnn.fc = nn.Linear(self.cnn.fc.in_features, num_classes)

    def forward(self, x):
        return self.cnn(x), None, None

### Grad-CAM Plus Plus

In [16]:
class GradCAMPlusPlus:
    def __init__(self, model, target_layer=None):
        self.model = model; self.gradients = None; self.activations = None
        if target_layer is None:
            target_layer = _get_gradcam_layer(model)
            print(f"   GradCAM++ target layer: {target_layer}")
        self._register_hooks(target_layer)

    def _register_hooks(self, target_layer):
        for name, module in self.model.named_modules():
            if name == target_layer:
                module.register_forward_hook(
                    lambda m, inp, out: setattr(self, 'activations', out))
                module.register_full_backward_hook(
                    lambda m, gi, go: setattr(self, 'gradients', go[0]))
                break

    def generate(self, img_tensor, target_class=None):
        img = img_tensor.unsqueeze(0) if img_tensor.dim() == 3 else img_tensor
        img = img.to(next(self.model.parameters()).device)
        self.model.zero_grad()
        with autocast(enabled=_USE_AMP):
            out = self.model(img)
        logits = out[0] if isinstance(out, tuple) else out
        if target_class is None: target_class = logits.argmax().item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(logits); one_hot[0, target_class] = 1
        logits.backward(gradient=one_hot, retain_graph=True)
        acts = self.activations.detach().float(); grads = self.gradients.detach().float()
        alpha_num = grads.pow(2)
        alpha_den  = 2 * grads.pow(2) + (acts * grads.pow(3)).sum(dim=(2,3), keepdim=True)
        alpha  = alpha_num / (alpha_den + 1e-8)
        weights = (alpha * F.relu(grads)).sum(dim=(2,3), keepdim=True)
        cam = F.relu((weights * acts).sum(dim=1, keepdim=True)).squeeze().cpu().numpy()
        cam = cam.astype(np.float32)
        if cam.max() - cam.min() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.astype(np.float32), target_class

### Attention Rollout

In [17]:
class AttentionRollout:
    def __init__(self, model, device):
        self.model = model; self.device = device; self.attentions = []
        for name, module in model.named_modules():
            if 'transformer' in name and hasattr(module, 'layers'):
                for layer in module.layers:
                    if hasattr(layer, 'self_attn'):
                        layer.self_attn.register_forward_hook(
                            lambda m, inp, out: self.attentions.append(m.attn_drop)
                            if hasattr(m, 'attn_drop') else None)

    def generate(self, img_tensor):
        self.attentions = []
        with torch.no_grad():
            with autocast(enabled=_USE_AMP):
                _ = self.model(img_tensor)
        if not self.attentions:
            h, w = img_tensor.shape[-2:]
            return np.ones((h, w)) / (h * w)
        result = torch.eye(197).to(self.device)
        for attn in self.attentions[:4]:
            result = torch.matmul(attn.mean(dim=1), result)
        cls_attn   = result[0, 0, 1:]
        g          = int(np.sqrt(cls_attn.shape[0]))
        attn_map   = cls_attn.reshape(g, g).cpu().numpy()
        attn_map   = zoom(attn_map, (224/g, 224/g), order=1)
        attn_map = attn_map.astype(np.float32)
        if attn_map.max() - attn_map.min() > 0:
            attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
        return attn_map.astype(np.float32)

### Fusion Saliency

In [18]:
class FusionSaliency:
    def __init__(self, model, device):
        self.model = model; self.device = device
        self.gradcam   = GradCAMPlusPlus(model)
        self.attention = AttentionRollout(model, device)

    def generate(self, img_tensor, original_image, save_path=None):
        if img_tensor.dim() == 3: img_tensor = img_tensor.unsqueeze(0)
        grad_cam_map, pred_class = self.gradcam.generate(img_tensor)
        attention_map            = self.attention.generate(img_tensor)

        def _resize(m, shape=(224,224)):
            if m.shape != shape:
                m = zoom(m, (shape[0]/m.shape[0], shape[1]/m.shape[1]), order=1)
            return m

        grad_cam_map  = _resize(grad_cam_map)
        attention_map = _resize(attention_map)

        intersection = grad_cam_map * attention_map
        weighted     = 0.6 * grad_cam_map + 0.4 * attention_map
        union        = np.maximum(grad_cam_map, attention_map)
        for m in [intersection, weighted, union]:
            if m.max() - m.min() > 0:
                m[:] = (m - m.min()) / (m.max() - m.min() + 1e-8)

        # img_np must be exactly (H, W, 3) float32 in [0, 1] matching the cam maps.
        # 'original_image' is the raw PIL image at whatever disk resolution — resize it.
        cam_h, cam_w = grad_cam_map.shape[:2]   # always 224×224 after _resize()
        if isinstance(original_image, Image.Image):
            img_np = np.array(
                original_image.convert('RGB').resize((cam_w, cam_h), Image.LANCZOS),
                dtype=np.float32) / 255.0
        else:
            img_np = np.asarray(original_image, dtype=np.float32) / 255.0
            if img_np.ndim == 3 and img_np.shape[-1] == 4:
                img_np = img_np[:, :, :3]
            if img_np.shape[:2] != (cam_h, cam_w):
                img_np = zoom(img_np,
                              (cam_h / img_np.shape[0], cam_w / img_np.shape[1], 1.0),
                              order=1)
        img_np = np.clip(img_np, 0.0, 1.0).astype(np.float32)

        results = dict(grad_cam=grad_cam_map, attention=attention_map,
                       intersection=intersection, weighted=weighted,
                       union=union, pred_class=pred_class, image_np=img_np)
        if save_path: self._visualize(results, save_path)
        return results

    def _visualize(self, r, save_path):
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        img = r['image_np']
        titles = ['Original Image', 'Grad-CAM++ (CNN Local)',
                  'Attention Rollout (ViT Global)',
                  'Fusion: Intersection', 'Fusion: Weighted (α=0.6)', 'Fusion: Union']
        maps   = [None, r['grad_cam'], r['attention'],
                  r['intersection'], r['weighted'], r['union']]
        for ax, title, m in zip(axes.flat, titles, maps):
            ax.imshow(show_cam_on_image(img, m, use_rgb=True) if m is not None else img)
            ax.set_title(title, fontsize=12, fontweight='bold'); ax.axis('off')
        plt.suptitle(f'Novel Fusion-Saliency  |  Predicted class: {r["pred_class"]}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()

### Image Loading

In [19]:
def load_test_images(test_path, num_samples, img_size, device):
    if not os.path.exists(test_path):
        print(f"Test path not found: {test_path}"); return [], []
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    dataset     = ImageFolder(test_path)
    class_names = dataset.classes
    images_info = []
    for path, label in dataset.samples[:num_samples]:
        try:
            img    = Image.open(path).convert('RGB')
            tensor = transform(img).to(device, non_blocking=True)
            images_info.append(dict(path=path, label=label,
                                    class_name=class_names[label],
                                    tensor=tensor, original=img))
        except Exception as e:
            print(f"Skip {path}: {e}")
    print(f"Loaded {len(images_info)} test images  | Classes: {class_names}")
    return images_info, class_names

### Run Analysis

In [20]:
def _save_overlay(img_np, cam_map, save_path, title='', class_name=''):
    """Save a 2-panel image: original | CAM overlay."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    overlay = show_cam_on_image(img_np, cam_map.astype(np.float32), use_rgb=True)
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_np);  axes[0].set_title('Original',  fontweight='bold'); axes[0].axis('off')
    axes[1].imshow(overlay); axes[1].set_title(title,        fontweight='bold'); axes[1].axis('off')
    plt.suptitle(f'Class: {class_name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()


def _save_comparison_grid(img_np, res, save_path, class_name=''):
    """Save a 3-panel comparison: Original | GradCAM++ | Attention Rollout."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    gc  = show_cam_on_image(img_np, res['grad_cam'].astype(np.float32),  use_rgb=True)
    att = show_cam_on_image(img_np, res['attention'].astype(np.float32), use_rgb=True)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, im, t in zip(axes,
                         [img_np, gc, att],
                         ['Original', 'Grad-CAM++ (CNN)', 'Attention Rollout (ViT)']):
        ax.imshow(im); ax.set_title(t, fontweight='bold'); ax.axis('off')
    plt.suptitle(f'Comparison Grid  |  Class: {class_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()


def _save_biological_validation(all_res, class_names, save_dir):
    """Per-class prediction accuracy bar chart + CSV summary."""
    import csv, collections
    os.makedirs(save_dir, exist_ok=True)
    correct = collections.Counter()
    total   = collections.Counter()
    for r in all_res:
        cls = r['cls']
        total[cls] += 1
        if isinstance(r['predicted'], int) and class_names[r['predicted']] == cls:
            correct[cls] += 1
    names = sorted(total.keys())
    accs  = [100 * correct[n] / total[n] if total[n] else 0 for n in names]
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(names, accs, color='steelblue', edgecolor='black')
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{acc:.0f}%', ha='center', fontweight='bold')
    ax.set_ylim(0, 115)
    ax.set_ylabel('Accuracy (%)'); ax.set_xlabel('Disease Class')
    ax.set_title('Biological Validation: Per-Class XAI Prediction Accuracy',
                 fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'class_accuracy_summary.png'),
                dpi=200, bbox_inches='tight', facecolor='white')
    plt.close()
    with open(os.path.join(save_dir, 'validation_summary.csv'), 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['Class', 'Total', 'Correct', 'Accuracy (%)'])
        for n in names:
            w.writerow([n, total[n], correct[n],
                        f'{100 * correct[n] / total[n]:.1f}' if total[n] else '0'])
    print(f"   Biological validation saved ({len(names)} classes)")


def main():
    print("STARTING XAIANALYSIS")

    model = load_any_model(XAIConfig.MODEL_PATH, XAIConfig.NUM_CLASSES, XAIConfig.DEVICE)
    if model is None: print("Model load failed!"); return

    test_images, class_names = load_test_images(
        XAIConfig.TEST_PATH, XAIConfig.NUM_SAMPLES,
        XAIConfig.IMG_SIZE, XAIConfig.DEVICE)
    if not test_images: print("No test images!"); return

    fusion  = FusionSaliency(model, XAIConfig.DEVICE)
    all_res = []
    print(f"\nGenerating explanations for {len(test_images)} samples...")

    for idx, info in enumerate(test_images):
        cls_name  = info['class_name']
        save_name = f"sample_{idx+1:02d}_{cls_name}"
        print(f"Sample {idx+1}: {cls_name}")
        try:
            res = fusion.generate(
                info['tensor'], info['original'],
                save_path=os.path.join(XAIConfig.OUTPUT_DIR,
                                       'Fusion_Saliency_Maps', f'{save_name}.png'))
            img_np = res['image_np']

            _save_overlay(
                img_np, res['grad_cam'],
                os.path.join(XAIConfig.OUTPUT_DIR, 'GradCAM_PlusPlus', f'{save_name}.png'),
                title='Grad-CAM++ (CNN Local)', class_name=cls_name)

            _save_overlay(
                img_np, res['attention'],
                os.path.join(XAIConfig.OUTPUT_DIR, 'Attention_Rollout', f'{save_name}.png'),
                title='Attention Rollout (ViT Global)', class_name=cls_name)

            _save_comparison_grid(
                img_np, res,
                os.path.join(XAIConfig.OUTPUT_DIR, 'Comparison_Grids', f'{save_name}.png'),
                class_name=cls_name)

            all_res.append(dict(cls=cls_name,
                                predicted=res['pred_class'], path=info['path']))
            print(f"   Predicted class index: {res['pred_class']}")
        except Exception as e:
            import traceback
            print(f"   Error: {e}")
            traceback.print_exc()

    _save_biological_validation(
        all_res, class_names,
        os.path.join(XAIConfig.OUTPUT_DIR, 'Biological_Validation'))

    output = dict(model_path=XAIConfig.MODEL_PATH,
                  num_samples=len(test_images), successful=len(all_res),
                  class_names=class_names, results=all_res)
    with open(os.path.join(XAIConfig.OUTPUT_DIR, 'xai_results.json'), 'w') as f:
        json.dump(output, f, indent=4)

    print(f"\nXAI ANALYSIS COMPLETE!")
    print(f"Results saved to: {XAIConfig.OUTPUT_DIR}")
    print(f"  Fusion_Saliency_Maps  : {len(all_res)} images")
    print(f"  GradCAM_PlusPlus      : {len(all_res)} images")
    print(f"  Attention_Rollout     : {len(all_res)} images")
    print(f"  Comparison_Grids      : {len(all_res)} images")
    print(f"  Biological_Validation : summary chart + CSV")
    return output

if __name__ == "__main__":
    try:
        results = main()
    except Exception as e:
        import traceback; print(f"\n {e}"); traceback.print_exc()


STARTING XAIANALYSIS

Loading model...
   State-dict keys (first 5): ['cnn_mask', 'transformer_mask', 'cnn.features.0.0.weight', 'cnn.features.0.1.weight', 'cnn.features.0.1.bias']
   Detected: XAIGuidedFusion (feature-selective hybrid)
   CNN backbone      : mobilenet_v2
   Transformer       : vit_base_patch16_224
   Selected CNN feats: 3
   Selected ViT feats: 231
   Fusion input dim  : 234  hidden: 234
   Skipped 4 head keys not in reconstruction: ['cnn.classifier.1.weight', 'cnn.classifier.1.bias', 'transformer.head.weight', 'transformer.head.bias']
   Weights loaded successfully
   Parameters: 88,108,838
Loaded 10 test images  | Classes: ['bacterial_blight', 'curl_virus', 'fussarium_wilt', 'healthy']
   GradCAM++ target layer: cnn.features.18

Generating explanations for 10 samples...
Sample 1: bacterial_blight
   Predicted class index: 0
Sample 2: bacterial_blight
   Predicted class index: 0
Sample 3: bacterial_blight
   Predicted class index: 0
Sample 4: bacterial_blight
   Pred